# Week 10 — Textual Analysis

**Focus:** What does financial text add, and when was the document actually available?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week10/week10_demos.ipynb)

## What this notebook is about

Financial text is data with vocabulary, context, authorship, and a publication clock. This notebook asks what text adds beyond simpler numeric information—and whether the documents were actually available when the decision was made.

The demonstrations move from bag-of-words and TF–IDF to finance-specific language, logistic prediction, document retrieval, similarity maps, timestamp leakage, and an incremental-signal test against a numeric baseline.

By the end, you should be able to:

- explain what bag-of-words and TF–IDF retain and discard;
- see why ordinary sentiment words can change meaning in finance;
- interpret cosine similarity as a retrieval rule rather than understanding;
- attach documents using availability time rather than the period they discuss; and
- test whether text improves a feasible baseline on genuinely later documents.

The tiny corpora are transparent teaching examples. Their scores show mechanisms and failure modes; they are not production estimates of document value.

### Course context

For a concise review and the optional textbook route, see [Week 10 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week10/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — Bag-of-words vs TF-IDF on a tiny finance corpus — <a href="#Demo-1-%E2%80%94-Bag-of-words-vs-TF-IDF-on-a-tiny-finance-corpus">Jupyter</a> · <a href="#scrollTo=md-272ee420df83">Colab</a></li>
<li>Demo 2 — A stylized Loughran-McDonald lexicon trap — <a href="#Demo-2-%E2%80%94-A-stylized-Loughran-McDonald-lexicon-trap">Jupyter</a> · <a href="#scrollTo=md-62df5689ae2c">Colab</a></li>
<li>Demo 3 — TF-IDF + logistic regression: the baseline that wins — <a href="#Demo-3-%E2%80%94-TF-IDF-%2B-logistic-regression%3A-the-baseline-that-wins">Jupyter</a> · <a href="#scrollTo=md-6f7104545dfb">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 4 — Lexical and concept-normalized retrieval — <a href="#Demo-4-%E2%80%94-Lexical-and-concept-normalized-retrieval">Jupyter</a> · <a href="#scrollTo=md-fb6b954ef122">Colab</a></li>
<li>Demo 5 — Doc-doc similarity heatmap — <a href="#Demo-5-%E2%80%94-Doc-doc-similarity-heatmap">Jupyter</a> · <a href="#scrollTo=md-2c22bf3386a0">Colab</a></li>
<li>Demo 6 — Timestamp leakage across repeated worlds — <a href="#Demo-6-%E2%80%94-Timestamp-leakage-across-repeated-worlds">Jupyter</a> · <a href="#scrollTo=md-f7b935e0f22d">Colab</a></li>
<li>Demo 7 — Incremental-signal test across chronological worlds — <a href="#Demo-7-%E2%80%94-Incremental-signal-test-across-chronological-worlds">Jupyter</a> · <a href="#scrollTo=md-5008d4430c09">Colab</a></li>
</ul>

## Project research-record checkpoint — proposal and plan freeze

Update the single `RESEARCH_RECORD.md` at the root of your project repository. This is the Weeks 10–11 proposal and working-baseline checkpoint, not a second notebook record. Before opening the final test lockbox:

1. **Evidence contract:** source, publication and availability time, exact vintage or document version, cutoff, and admissible span.
2. **Baseline:** a feasible numeric, lexical, retrieval, or other simple comparator and why it is fair.
3. **Frozen procedure:** candidate system, development and selection roles, final lockbox, search space, primary metric and denominator.
4. **Incremental test:** what the new evidence must add beyond the baseline and the pass, fail, or kill criterion.
5. **Freeze:** plan-freeze date and commit hash from before the final test is opened.

If the plan changes afterward, do not overwrite the frozen version: append a dated log entry, state what evidence had been seen, and adjust the claim or trial count.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)


def finish_plot():
    """Show figures interactively and close them in noninteractive checks."""
    if "agg" not in plt.get_backend().lower():
        plt.show()
    plt.close()

### Demo 1 — Bag-of-words vs TF-IDF on a tiny finance corpus

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
corpus = [
    "the company reported strong revenue growth this quarter",
    "the company reported a tax liability this quarter",
    "the company recorded depreciation and capital costs this quarter",
    "the company filed a litigation update this quarter",
    "the firm raised earnings guidance this quarter",
    "the firm cut its dividend this quarter due to weak demand",
]
cv = CountVectorizer().fit(corpus)
tv = TfidfVectorizer(norm="l2").fit(corpus)
counts = cv.transform(corpus).toarray()
tfidf = tv.transform(corpus).toarray()
vocab = cv.get_feature_names_out()

for doc_idx in [0, 1]:
    nonzero = np.flatnonzero(tfidf[doc_idx])
    order = nonzero[np.argsort(-tfidf[doc_idx, nonzero])]
    print(f"\nDocument: '{corpus[doc_idx]}'")
    print(f"{'term':<14} {'count':>6} {'tfidf':>8}")
    for k in order:
        print(f"{vocab[k]:<14} {counts[doc_idx, k]:>6d} {tfidf[doc_idx, k]:>8.3f}")
print(f"Vocabulary size: {tfidf.shape[1]}")
print(f"TF-IDF density: {np.count_nonzero(tfidf) / tfidf.size:.3f}")
# Note how 'the', 'company', 'this', 'quarter' have high counts but low TF-IDF;
# distinctive terms like 'strong', 'revenue', 'growth' rise under TF-IDF.


def compare_collisions(pairs):
    """Compare labeled sentence pairs under three declared TF-IDF instruments."""
    documents = [sentence for _, left, right in pairs for sentence in (left, right)]
    representations = {
        "word unigram": dict(analyzer="word", ngram_range=(1, 1), norm="l2"),
        "word unigram + bigram": dict(analyzer="word", ngram_range=(1, 2), norm="l2"),
        "character 3-5 gram": dict(analyzer="char", ngram_range=(3, 5), norm="l2"),
    }
    rows = []
    print("\nCollision cosine similarities")
    print(f"{'pair':<18} {'representation':<24} {'cosine':>8}")
    for representation, settings in representations.items():
        matrix = TfidfVectorizer(**settings).fit_transform(documents)
        for pair_index, (label, _, _) in enumerate(pairs):
            left = matrix[2 * pair_index]
            right = matrix[2 * pair_index + 1]
            similarity = float(cosine_similarity(left, right)[0, 0])
            rows.append((label, representation, similarity))
            print(f"{label:<18} {representation:<24} {similarity:>8.3f}")
    return rows

### Mini-project: create a collision

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 1 — Bag-of-words versus TF-IDF**, then call its
`compare_collisions` helper on the three sentence pairs below.

Construct three sentence pairs:

1. order reversal;
2. negation;
3. financial paraphrase.

Compare word, bigram, and character cosine similarity. Explain one improvement
and one remaining collision.

**Research record:** the three sentence pairs; word, bigram, and character cosine similarity for each; the improvement you found and the collision that survived it.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 2 — A stylized Loughran-McDonald lexicon trap

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
generic_negative = {
    "liability", "tax", "cost", "capital", "depreciation", "foreign",
    "decline", "impairment", "litigation", "weak",
}
finance_negative = {  # LM-style: technical-but-neutral terms removed
    "decline", "impairment", "litigation", "weak",
}

neutral_sentences = [
    "the company recorded a tax liability",
    "depreciation and capital expenditure were reported",
    "the filing described foreign tax rules",
    "the board reviewed the cost of capital",
    "capital and liabilities appear on the balance sheet",
    "depreciation changed the tax basis",
]
negative_sentences = [
    "revenue decline reflected weak demand",
    "the company recorded an impairment",
    "litigation created a material loss",
]


def score(text, lex):
    toks = text.lower().split()
    neg = sum(1 for t in toks if t.strip(",.") in lex)
    return -neg


gen = np.array([score(s, generic_negative) for s in neutral_sentences])
fin = np.array([score(s, finance_negative) for s in neutral_sentences])
gen_negative = np.array([score(s, generic_negative) for s in negative_sentences])
fin_negative = np.array([score(s, finance_negative) for s in negative_sentences])
print(f"\nNeutral finance sentences: {len(neutral_sentences)}")
print(f"  Generic lexicon scores < 0: {(gen < 0).sum()}  (false alarms for negative sentiment)")
print(f"  Finance lexicon scores < 0: {(fin < 0).sum()}")
print(f"Constructed negative sentences: {len(negative_sentences)}")
print(f"  Generic lexicon detects: {(gen_negative < 0).sum()}")
print(f"  Finance lexicon detects: {(fin_negative < 0).sum()}")
print("Generic lexicon measures accounting jargon, not mood.")

### Demo 3 — TF-IDF + logistic regression: the baseline that wins

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
pos = [
    "revenue grew strongly this quarter",
    "the firm raised earnings guidance",
    "management lifted its outlook for next year",
    "operating margin expanded versus prior period",
    "the company exceeded analyst expectations",
    "robust demand drove record revenue",
    "earnings beat consensus by a wide margin",
    "free cash flow grew sharply this quarter",
    "the firm announced a substantial buyback",
    "results were ahead of guidance",
]
neg = [
    "the company missed revenue expectations",
    "guidance was cut for the second time",
    "margins contracted on weak demand",
    "an impairment charge weighed on results",
    "litigation reserves were materially raised",
    "the firm reported a sharp decline in sales",
    "earnings fell short of consensus",
    "free cash flow turned negative this quarter",
    "the company warned of further weakness ahead",
    "results came in below the low end of guidance",
]
X = pos + neg
y = np.array([1] * len(pos) + [0] * len(neg))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=87,
                                      stratify=y)
pipe = TfidfVectorizer(min_df=1, norm="l2")
Xtr_v = pipe.fit_transform(Xtr)
Xte_v = pipe.transform(Xte)
clf = LogisticRegression(max_iter=1000).fit(Xtr_v, ytr)
pred = clf.predict(Xte_v)
acc = accuracy_score(yte, pred)
majority = max((yte == 1).mean(), (yte == 0).mean())
print(f"\nTF-IDF + logistic regression: test accuracy = {acc:.3f} vs "
      f"majority baseline {majority:.3f}")
print("Confusion matrix:")
print(confusion_matrix(yte, pred))
# This is the line FinBERT must clear before deployment is justified.

### Mini-project: inspect the fitted instrument

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demos 1 and 3 — Bag-of-words versus TF-IDF and the logistic baseline**.

Using `week10_demos.ipynb`:

1. fit counts and TF-IDF on training documents only;
2. print the highest-weight terms in two documents;
3. inspect vocabulary size and sparsity;
4. fit the logistic baseline and report its held-out confusion matrix.

**Research record:** the training documents the vectorizer saw; vocabulary size and sparsity; the highest-weight terms in two documents; the baseline's held-out confusion matrix.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Mini-project: validate the domain repair

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demos 2 and 3 — Finance-lexicon trap and the logistic baseline**.

1. record neutral false flags under both illustrative lexicons;
2. verify detection on the three constructed negative sentences;
3. report the train-fitted baseline's held-out confusion matrix;
4. state one conclusion and one boundary of the comparison.

**Research record:** both false-flag counts; both negative-detection counts; the baseline confusion matrix; the domain-repair conclusion and its small-sample boundary.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — Lexical and concept-normalized retrieval

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
library = [
    "the company raised its earnings guidance for the year",
    "the firm lifted its revenue outlook citing strong demand",
    "the board declared the regular quarterly dividend",
    "operating margins improved on cost discipline",
    "the firm faces a new class-action litigation",
    "headcount grew by twelve percent year over year",
    "free cash flow declined due to working capital changes",
    "the firm announced a share buyback program",
]
query = "did the company change its earnings or revenue guidance"

# The library defines the fitted lexical instrument. A later query may be
# transformed by that instrument, but it must not alter the vocabulary or IDF.
vec = TfidfVectorizer(norm="l2").fit(library)
L = vec.transform(library)
q = vec.transform([query])
sims = cosine_similarity(q, L).ravel()

order = np.argsort(-sims)
print("Top retrieval results for query:")
print(f"  '{query}'")
for k in order[:4]:
    print(f"  {sims[k]:.3f}  {library[k]}")

# Judgments turn the example into a small retrieval evaluation.
relevant = {0, 1}
k = 3
top_k = order[:k]
recall_at_k = len(set(top_k) & relevant) / len(relevant)
first_relevant_rank = next(i for i, idx in enumerate(order, start=1)
                           if idx in relevant)
print(f"Recall@{k}: {recall_at_k:.3f}")
print(f"Reciprocal rank: {1 / first_relevant_rank:.3f}")

# A hand-built semantic normalization baseline supplies the second method used
# in the mini-project. It is not a production dense encoder.
concepts = {
    "raised": "increase_guidance", "lifted": "increase_guidance",
    "guidance": "outlook", "earnings": "results", "revenue": "results",
}


def normalize_concepts(text):
    return " ".join(concepts.get(token, token) for token in text.lower().split())


normalized_library = [normalize_concepts(text) for text in library]
normalized_query = normalize_concepts(query)
concept_vec = TfidfVectorizer(norm="l2").fit(normalized_library)
concept_L = concept_vec.transform(normalized_library)
concept_q = concept_vec.transform([normalized_query])
concept_sims = cosine_similarity(concept_q, concept_L).ravel()
concept_order = np.argsort(-concept_sims)
concept_top_k = concept_order[:k]
concept_recall = len(set(concept_top_k) & relevant) / len(relevant)
concept_first_rank = next(i for i, idx in enumerate(concept_order, start=1)
                          if idx in relevant)
print(f"Concept-normalized Recall@{k}: {concept_recall:.3f}")
print(f"Concept-normalized reciprocal rank: {1 / concept_first_rank:.3f}")

### Mini-project: build a judged query set

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 4 — Lexical and concept-normalized retrieval**, then replace its queries with your judged set.

1. Write five questions about a small filing library; include one negated or
   time-scoped question.
2. Mark relevant passages before comparing retrievers.
3. Run lexical retrieval and one normalized or dense alternative.
4. Report Recall@3 and mean reciprocal rank (MRR).

**Research record:** the five questions and the passages marked relevant before retrieval ran; Recall@3 and MRR for each retriever; the query on which the methods fail differently.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 5 — Doc-doc similarity heatmap

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
M = cosine_similarity(L, L)
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1)
ax.set_title("Document-document cosine similarity")
short_labels = ["guidance", "outlook", "dividend", "margin",
                "litigation", "headcount", "cash flow", "buyback"]
ax.set_xticks(range(len(library)), short_labels, rotation=45, ha="right")
ax.set_yticks(range(len(library)), short_labels)
fig.colorbar(im, ax=ax)
fig.tight_layout()
finish_plot()
# Bright off-diagonal cells reveal which docs resemble each other.

### Demo 6 — Timestamp leakage across repeated worlds

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# The same released tone is aligned first with an already-realized interval and
# then with a genuinely later interval. Repetition separates the mechanism from
# one lucky simulation draw.
timing_records = []
for seed in range(30):
    timing_rng = np.random.default_rng(6100 + seed)
    n = 800
    tone = timing_rng.standard_normal(n)
    past_ret = 0.9 * tone + 0.6 * timing_rng.standard_normal(n)
    future_ret = 0.08 * tone + timing_rng.standard_normal(n)
    timing_records.append((
        np.mean(np.sign(tone) == np.sign(past_ret)),
        np.mean(np.sign(tone) == np.sign(future_ret)),
    ))

leaked_acc = np.median([row[0] for row in timing_records])
admissible_acc = np.median([row[1] for row in timing_records])
print(f"Median leaked accuracy across 30 worlds    : {leaked_acc:.3f}")
print(f"Median admissible accuracy across 30 worlds: {admissible_acc:.3f}")

# A miniature passport makes the operational timing rule executable.
public_day = 35.00
feature_ready_day = 35.08
decision_day = 35.10
target_start_day = decision_day
public_margin = decision_day - public_day
usable_margin = decision_day - feature_ready_day
assert public_margin >= 0
assert usable_margin >= 0
assert target_start_day >= decision_day
print(f"Example public margin: {public_margin:.2f} days")
print(f"Example usable margin: {usable_margin:.2f} days")

### Mini-project: draw the passport

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 6 — Timestamp leakage**, then use its availability logic to passport one real document.

Choose one filing or earnings call and record:

1. subject period;
2. public timestamp and timezone;
3. version;
4. feature-ready timestamp or declared latency policy;
5. earliest usable decision and target interval.

**Research record:** subject period; public timestamp and timezone; version; feature-ready timestamp or latency assumption; earliest usable decision time; target interval; both availability margins and the execution convention.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 7 — Incremental-signal test across chronological worlds

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
positive = ["raised guidance", "demand strengthened", "orders improved"]
negative = ["cut guidance", "demand weakened", "orders declined"]
neutral = ["results were reported", "management held a call", "the filing was released"]
increment_records = []
for seed in range(12):
    world_rng = np.random.default_rng(6300 + seed)
    n = 1500
    numeric = world_rng.standard_normal(n)
    tone = world_rng.standard_normal(n)
    docs = []
    for value in tone:
        phrases = positive if value > 0.35 else negative if value < -0.35 else neutral
        phrase = world_rng.choice(phrases)
        docs.append(f"company {world_rng.integers(100)} {phrase}")
    latent = 0.75 * numeric + 0.22 * tone + world_rng.standard_normal(n)
    y = (latent > 0).astype(int)
    train = np.arange(0, 800)
    valid = np.arange(800, 1000)
    test = np.arange(1000, 1500)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, norm="l2").fit([docs[i] for i in train])
    Xtext = vec.transform(docs)

    for use_text in [False, True]:
        candidates = []
        for C in [0.1, 1, 10]:
            Xtr = numeric[train, None] if not use_text else hstack([numeric[train, None], Xtext[train]])
            Xv = numeric[valid, None] if not use_text else hstack([numeric[valid, None], Xtext[valid]])
            model = LogisticRegression(C=C, max_iter=1000).fit(Xtr, y[train])
            candidates.append((log_loss(y[valid], model.predict_proba(Xv)[:, 1]), C))
        selected_C = min(candidates)[1]
        development = np.arange(0, 1000)
        Xdev = numeric[development, None] if not use_text else hstack([numeric[development, None], Xtext[development]])
        Xt = numeric[test, None] if not use_text else hstack([numeric[test, None], Xtext[test]])
        model = LogisticRegression(C=selected_C, max_iter=1000).fit(Xdev, y[development])
        probability = model.predict_proba(Xt)[:, 1]
        increment_records.append((
            seed,
            "numeric + text" if use_text else "numeric only",
            log_loss(y[test], probability),
            accuracy_score(y[test], probability >= 0.5),
        ))

improvements = []
for seed in range(12):
    numeric_loss = next(row[2] for row in increment_records
                        if row[0] == seed and row[1] == "numeric only")
    text_loss = next(row[2] for row in increment_records
                     if row[0] == seed and row[1] == "numeric + text")
    improvements.append(numeric_loss - text_loss)
print(f"Median paired log-loss improvement: {np.median(improvements):+.4f}")
print(f"Range across worlds: {min(improvements):+.4f} to {max(improvements):+.4f}")
print(f"Worlds improved: {np.sum(np.array(improvements) > 0)} of {len(improvements)}")

### Mini-project: test incremental text value

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 7 — Incremental-signal test across chronological worlds**.

1. report the paired log-loss improvement in each world;
2. summarize the median, range, and number of positive worlds;
3. state whether the current pipeline has shown stable incremental value;
4. name one development-only follow-up and the later evidence it would require.

**Research record:** the twelve paired improvements; median and range; positive-world count; the bounded conclusion; the frozen follow-up and later test requirement.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

## Where this leaves us

Text features are representations learned or defined from a dated document population. They can add information, leak future language, or merely restate a numeric baseline. Only an aligned incremental test distinguishes those possibilities.

Week 11 treats retrieval, generation, refusal, and authority as parts of one auditable language-model system.